# Task 2: LoRA Fine-Tuning of Llama 3.2 1B for Named Entity Recognition

## Internship Experiment 2

This notebook performs parameter-efficient fine-tuning (LoRA) on Llama 3.2 1B for a token classification (NER) task.

### Objectives

- Load and validate the dataset
- Perform tokenization and label alignment
- Configure LoRA adapters
- Fine-tune the model
- Evaluate model performance
- Compare with baseline results

In [1]:
!pip install -q transformers datasets evaluate seqeval peft bitsandbytes accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00


# 1. Environment Setup

In [2]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from transformers import set_seed

set_seed(42)

print("Random seed set to 42")

Random seed set to 42


In [4]:
import os
import json
import numpy as np
import torch
import evaluate

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [5]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

GPU: Tesla T4


# 2. Dataset Loading and Validation

The dataset is stored in CoNLL format where each token is associated with a NER label.

This section:
- Loads the train, validation and test splits
- Validates dataset integrity
- Inspects example samples

In [6]:
train_path = "/content/train.txt"
valid_path = "/content/valid.txt"
test_path = "/content/test.txt"

print("Train file:", train_path)
print("Validation file:", valid_path)
print("Test file:", test_path)

Train file: /content/train.txt
Validation file: /content/valid.txt
Test file: /content/test.txt


In [7]:
def preview_file(path, n=10):
    print(f"\nPreviewing: {path}\n")

    with open(path, "r", encoding="utf-8") as f:
        for i in range(n):
            print(repr(f.readline()))

preview_file(train_path, 10)


Previewing: /content/train.txt

'What\tO\n'
'is\tO\n'
'here\tO\n'
'called\tO\n'
'controlled\tB-long\n'
'natural\tI-long\n'
'language\tI-long\n'
'(\tO\n'
'CNL\tB-short\n'
')\tO\n'


In [8]:
def read_conll_file(filepath):

    sentences = []
    tags = []

    current_sentence = []
    current_tags = []

    with open(filepath, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if line == "":

                if current_sentence:
                    sentences.append(current_sentence)
                    tags.append(current_tags)

                    current_sentence = []
                    current_tags = []

            else:

                parts = line.split()

                if len(parts) >= 2:

                    token = parts[0]
                    tag = parts[-1]

                    current_sentence.append(token)
                    current_tags.append(tag)

    if current_sentence:
        sentences.append(current_sentence)
        tags.append(current_tags)

    return sentences, tags

In [9]:
train_sentences, train_tags = read_conll_file(train_path)
valid_sentences, valid_tags = read_conll_file(valid_path)
test_sentences, test_tags = read_conll_file(test_path)

print("Train sentences:", len(train_sentences))
print("Validation sentences:", len(valid_sentences))
print("Test sentences:", len(test_sentences))

Train sentences: 14006
Validation sentences: 1717
Test sentences: 1749


In [10]:
print("Example sentence:\n")
print(train_sentences[0])

print("\nExample labels:\n")
print(train_tags[0])

Example sentence:

['What', 'is', 'here', 'called', 'controlled', 'natural', 'language', '(', 'CNL', ')', 'has', 'traditionally', 'been', 'given', 'many', 'different', 'names', '.']

Example labels:

['O', 'O', 'O', 'O', 'B-long', 'I-long', 'I-long', 'O', 'B-short', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [11]:
def validate_dataset(sentences, tags, name):

    mismatched = 0
    empty_sentences = 0

    for s, t in zip(sentences, tags):

        if len(s) != len(t):
            mismatched += 1

        if len(s) == 0:
            empty_sentences += 1

    print(f"{name} total samples: {len(sentences)}")
    print(f"{name} mismatched token/tag sequences: {mismatched}")
    print(f"{name} empty sentences: {empty_sentences}")

In [12]:
validate_dataset(train_sentences, train_tags, "TRAIN")
validate_dataset(valid_sentences, valid_tags, "VALID")
validate_dataset(test_sentences, test_tags, "TEST")

TRAIN total samples: 14006
TRAIN mismatched token/tag sequences: 0
TRAIN empty sentences: 0
VALID total samples: 1717
VALID mismatched token/tag sequences: 0
VALID empty sentences: 0
TEST total samples: 1749
TEST mismatched token/tag sequences: 0
TEST empty sentences: 0


# 3. Label Processing

NER labels are converted into numerical IDs so they can be used by the model.

The mappings are:
- label → id
- id → label

In [13]:
all_labels = sorted(
    list(
        set(
            label.lower()
            for split_tags in [train_tags, valid_tags, test_tags]
            for tag_list in split_tags
            for label in tag_list
        )
    )
)

print("Labels:")
print(all_labels)

print("\nNumber of labels:")
print(len(all_labels))

Labels:
['b-long', 'b-short', 'i-long', 'i-short', 'o']

Number of labels:
5


In [14]:
label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for label, i in label2id.items()}

print("label2id:")
print(label2id)

print("\nid2label:")
print(id2label)

label2id:
{'b-long': 0, 'b-short': 1, 'i-long': 2, 'i-short': 3, 'o': 4}

id2label:
{0: 'b-long', 1: 'b-short', 2: 'i-long', 3: 'i-short', 4: 'o'}


In [15]:
print("\nDataset Summary")
print("-" * 40)

print(f"Train Samples      : {len(train_sentences)}")
print(f"Validation Samples : {len(valid_sentences)}")
print(f"Test Samples       : {len(test_sentences)}")

print(f"\nNumber of Labels   : {len(all_labels)}")
print(f"Labels             : {all_labels}")


Dataset Summary
----------------------------------------
Train Samples      : 14006
Validation Samples : 1717
Test Samples       : 1749

Number of Labels   : 5
Labels             : ['b-long', 'b-short', 'i-long', 'i-short', 'o']


# 4. Hugging Face Dataset Creation

The raw Python lists are converted into Hugging Face Dataset objects to enable efficient preprocessing, batching, and model training.

In [16]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_dict({
        "tokens": train_sentences,
        "ner_tags": train_tags
    }),

    "validation": Dataset.from_dict({
        "tokens": valid_sentences,
        "ner_tags": valid_tags
    }),

    "test": Dataset.from_dict({
        "tokens": test_sentences,
        "ner_tags": test_tags
    })
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14006
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1717
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1749
    })
})


In [17]:
print("\nFirst Training Example:\n")

print(dataset["train"][0])


First Training Example:

{'tokens': ['What', 'is', 'here', 'called', 'controlled', 'natural', 'language', '(', 'CNL', ')', 'has', 'traditionally', 'been', 'given', 'many', 'different', 'names', '.'], 'ner_tags': ['O', 'O', 'O', 'O', 'B-long', 'I-long', 'I-long', 'O', 'B-short', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}
